# Day G3 — 영상·신호·EHR의 생성형 AI: 증강, 합성 데이터, 프라이버시

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/G-ai-biomed/day-G3.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/G-ai-biomed/day-G3.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 합성 EHR 표를 세 축으로 점검하기

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.mixture import GaussianMixture
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
COLS = ["age", "bmi", "sbp", "hba1c"]

def make_cohort(n):
    """가상 EHR 표: 나이, BMI, 수축기 혈압, HbA1c와 가상 결과(outcome)."""
    age = rng.normal(62, 10, n)
    bmi = rng.normal(26, 4, n)
    sbp = 90 + 0.6 * age + 0.8 * bmi + rng.normal(0, 10, n)
    hba1c = 4.5 + 0.02 * age + 0.06 * bmi + rng.normal(0, 0.6, n)
    logit = -14 + 0.08 * age + 0.05 * sbp + 0.6 * hba1c
    outcome = rng.random(n) < 1 / (1 + np.exp(-logit))
    return pd.DataFrame({"age": age, "bmi": bmi, "sbp": sbp, "hba1c": hba1c, "outcome": outcome.astype(int)})

real = make_cohort(1200)
train, holdout = train_test_split(real, test_size=0.5, random_state=0, stratify=real["outcome"])

def synthesize(df, n_components, n_out):
    """결과 군마다 GaussianMixture를 따로 맞추고, 원래 비율대로 새 행을 뽑는다."""
    parts = []
    for y, g in df.groupby("outcome"):
        gm = GaussianMixture(n_components=min(n_components, len(g)), covariance_type="full",
                             reg_covar=1e-6, random_state=0).fit(g[COLS])
        k = int(round(n_out * len(g) / len(df)))
        s, _ = gm.sample(k)
        parts.append(pd.DataFrame(s, columns=COLS).assign(outcome=y))
    return pd.concat(parts, ignore_index=True)

def report(name, synth):
    # 1) 충실도: 열마다 분포 비교(KS 통계량, 작을수록 비슷), 상관 행렬 차이
    ks = [stats.ks_2samp(train[c], synth[c]).statistic for c in COLS]
    corr_gap = np.abs(train[COLS].corr().values - synth[COLS].corr().values).max()
    # 2) 유용성: 합성으로 학습 → 실제 holdout으로 평가(TSTR)
    m = LogisticRegression(max_iter=1000).fit(synth[COLS], synth["outcome"])
    tstr = roc_auc_score(holdout["outcome"], m.predict_proba(holdout[COLS])[:, 1])
    # 3) 탐지 검사: 학습에 안 쓴 실제(holdout)와 합성을 구별하는 분류기의 AUC
    #    0.5에 가까울수록 구별이 어렵다. (학습 행과 비교하면 '복사본' 때문에 값이 왜곡된다)
    both = pd.concat([holdout[COLS].assign(is_syn=0), synth[COLS].assign(is_syn=1)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
    p = cross_val_predict(RandomForestClassifier(n_estimators=200, random_state=0),
                          both[COLS], both["is_syn"], cv=cv, method="predict_proba")[:, 1]
    det = roc_auc_score(both["is_syn"], p)
    # 4) 프라이버시: 합성 행에서 가장 가까운 학습 행까지의 거리(DCR)
    sc = StandardScaler().fit(train[COLS])
    A, H, S = (sc.transform(d[COLS]) for d in (train, holdout, synth))
    dcr_syn = np.median(np.sqrt(((S[:, None] - A[None]) ** 2).sum(-1)).min(1))
    dcr_hold = np.median(np.sqrt(((H[:, None] - A[None]) ** 2).sum(-1)).min(1))
    print(f"{name:<22} KS최대 {max(ks):.3f} | 상관차 {corr_gap:.3f} | TSTR AUC {tstr:.3f} | "
          f"탐지 AUC {det:.3f} | DCR 합성 {dcr_syn:.3f} vs holdout {dcr_hold:.3f}")

m = LogisticRegression(max_iter=1000).fit(train[COLS], train["outcome"])
print("기준: 실제로 학습 → 실제 평가(TRTR) AUC",
      round(roc_auc_score(holdout["outcome"], m.predict_proba(holdout[COLS])[:, 1]), 3))
report("GMM 성분 3개", synthesize(train, 3, len(train)))
report("GMM 성분 300개(과적합)", synthesize(train, 300, len(train)))

### 예시 3 — 차분 프라이버시: 평균 하나를 공개할 때

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
# 차분 프라이버시(differential privacy)의 가장 단순한 예: 평균에 라플라스 잡음 더하기
age = rng.normal(62, 10, 200).clip(18, 90)          # 가상 나이 200명, 18–90세로 자름
sens = (90 - 18) / len(age)                         # 한 사람이 바뀔 때 평균이 최대로 움직이는 폭
print(f"실제 평균 나이 {age.mean():.2f}")
for eps in [0.1, 1.0, 10.0]:                        # ε가 작을수록 강한 보호, 큰 잡음
    noisy = [age.mean() + rng.laplace(0, sens / eps) for _ in range(1000)]
    print(f"ε = {eps:>4}: 공개되는 평균의 표준편차 {np.std(noisy):.2f}세")